# Baseline Modeling (7개 모델 비교)

데이터 로드 → 전처리(클리닝/이상치/집계/스케일링) → 7개 모델 Baseline 비교 및 Feature Importance

**목적**: 다양한 모델을 기본 파라미터로 돌려 RMSE를 한눈에 비교하고, 유망 모델을 선별.

**모델**: LightGBM, XGBoost, CatBoost, RandomForest(cuML GPU), RidgeCV, LassoCV, ElasticNetCV

## 1. 환경 설정 및 데이터 로드

In [ ]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    if not os.path.exists("/content/project/2_preprocessing/cleaning.py"):
        os.system("gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip")
        os.system("unzip -qo /content/preprocessing.zip -d /content/project")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

# ─── 공통 import ───
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from tqdm.auto import tqdm

# 트리 모델
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
# RandomForest: cuML GPU 가속 (fallback → sklearn CPU)
try:
    from cuml.ensemble import RandomForestRegressor
    CUML_RF = True
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    CUML_RF = False

# 선형 모델 (CV 내장 = alpha 자동 탐색)
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

# 스케일링 / 평가
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error

# 프로젝트 utils
from utils.config import PROJECT_ROOT, SEED, TARGET_COL
from utils.data import load_all, get_feat_cols, split_xs
from utils.aggregate import merge_with_target
from utils.evaluate import evaluate, postprocess, compare_models

# 전처리 모듈
sys.path.insert(0, os.path.join(PROJECT_ROOT, "2_preprocessing"))
from cleaning import run_cleaning
from outlier import run_outlier_treatment
from aggregation import run_aggregation

# ─── 데이터 로드 ───
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
ys_train = ys["train"]
ys_val = ys["validation"]

## 2. 전처리 파이프라인 (클리닝 → 이상치 → 집계)

In [ ]:
# --- Step 1: 클리닝 (상수/고결측/중복 제거 + 결측 imputation) ---
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    const_threshold=1e-6,      # std가 이 값 이하인 feature 제거
    missing_threshold=0.5,     # 결측률이 이 비율 이상인 feature 제거
    remove_duplicates=True,    # 값이 완전히 동일한 중복 컬럼 제거
)

In [ ]:
# --- Step 2: 이상치 처리 (Winsorization) ---
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    lower_pct=0.01,   # 하위 분위수 경계 (이 미만 값을 클리핑)
    upper_pct=0.99,   # 상위 분위수 경계 (이 초과 값을 클리핑)
)

In [ ]:
# --- Step 3: Die → Unit 집계 ---
unit_train, unit_val, unit_test, unit_feat_cols = run_aggregation(
    xs_train, xs_val, xs_test, clean_cols,
    agg_funcs=["mean", "std", "min", "max"],  # die 4개를 unit으로 집계할 통계량
    use_position_pivot=False,                   # True면 position별 피벗 feature도 추가
    save_csv=False,                             # baseline에서는 CSV 저장 안 함
)

## 3. Target Merge 및 학습 데이터 준비

In [ ]:
# Target merge (train / val / test)
X_train, y_train = merge_with_target(unit_train, split="train")
X_val, y_val = merge_with_target(unit_val, split="validation")
X_test, y_test = merge_with_target(unit_test, split="test")

# NaN 확인
assert X_train.isnull().sum().sum() == 0, "X_train에 NaN 존재!"
assert X_val.isnull().sum().sum() == 0, "X_val에 NaN 존재!"
assert X_test.isnull().sum().sum() == 0, "X_test에 NaN 존재!"

# RobustScaler (선형/딥러닝 모델용이지만, 트리 모델에 영향 없으므로 1벌로 통일)
scaler = RobustScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"\ny_train: mean={y_train.mean():.6f}, zero={(y_train==0).mean()*100:.1f}%")
print(f"y_val:   mean={y_val.mean():.6f}, zero={(y_val==0).mean()*100:.1f}%")
print(f"y_test:  mean={y_test.mean():.6f}, zero={(y_test==0).mean()*100:.1f}%")

# 단순 baseline (비교 기준선)
print(f"\n--- 단순 Baseline ---")
evaluate(y_val, np.zeros(len(y_val)), label="all-zero")
evaluate(y_val, np.full(len(y_val), y_train.mean()), label="train-mean")

In [ ]:
# 임시 --------------------------------------------------------------------------
# ===== 1/10 샘플링 (실험용) =====
SAMPLE_FRAC = 0.1
sample_idx = X_train_s.sample(frac=SAMPLE_FRAC, random_state=SEED).index
X_train_s = X_train_s.loc[sample_idx].reset_index(drop=True)
y_train = y_train.loc[sample_idx].reset_index(drop=True)
print(f"샘플링: {len(sample_idx)} → {len(X_train_s)} ({SAMPLE_FRAC*100:.0f}%)")

## 4. 7개 모델 Baseline 비교 (Train → Val/Test 평가)

트리 4개 + 선형 3개를 **기본 파라미터**로 학습하여 Val/Test RMSE를 비교

- **학습**: Train set으로 학습 (LightGBM/XGBoost/CatBoost는 Val set으로 early stopping)
- **선형 모델**: RidgeCV/LassoCV/ElasticNetCV로 alpha 자동 탐색 (공정한 디폴트)
- **RandomForest**: cuML GPU 가속 (미설치 시 sklearn CPU fallback)

In [ ]:
# ─── 7개 모델 정의 ───
_rf_params = dict(n_estimators=100, random_state=SEED, max_depth=16)
if not CUML_RF:
    _rf_params["n_jobs"] = -1

models = {
    "LightGBM": lgb.LGBMRegressor(
        random_state=SEED, n_jobs=-1, verbose=-1,
    ),
    "XGBoost": xgb.XGBRegressor(
        random_state=SEED, n_jobs=-1, verbosity=0,
        early_stopping_rounds=50,
    ),
    "CatBoost": CatBoostRegressor(
        random_seed=SEED, verbose=0,
        early_stopping_rounds=50,
    ),
    "RandomForest": RandomForestRegressor(**_rf_params),
    "RidgeCV": RidgeCV(
        alphas=np.logspace(-6, 6, 50),
    ),
    "LassoCV": LassoCV(
        alphas=np.logspace(-6, 2, 50), max_iter=5000, random_state=SEED, n_jobs=-1,
    ),
    "ElasticNetCV": ElasticNetCV(
        alphas=np.logspace(-6, 2, 50), l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99],
        max_iter=5000, random_state=SEED, n_jobs=-1,
    ),
}

print(f"모델 {len(models)}개 정의 완료")
if CUML_RF:
    print("  → RandomForest: cuML GPU 가속")
else:
    print("  → RandomForest: sklearn CPU")

In [ ]:
# ─── 7개 모델: Train 학습 → Val/Test 평가 ───
results = {}
trained_models = {}

pbar = tqdm(models.items(), desc="모델 학습", unit="model")
for name, model in pbar:
    pbar.set_postfix_str(name)

    # 학습 (LightGBM/XGBoost/CatBoost는 val로 early stopping)
    if name == "LightGBM":
        model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    elif name == "XGBoost":
        model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)
    elif name == "CatBoost":
        model.fit(X_train_s, y_train, eval_set=(X_val_s, y_val))
    else:
        model.fit(X_train_s, y_train)

    # Val / Test 예측 + 평가
    pred_val = postprocess(model.predict(X_val_s))
    pred_test = postprocess(model.predict(X_test_s))
    val_rmse = evaluate(y_val, pred_val, label=f"{name} (val)")
    test_rmse = evaluate(y_test, pred_test, label=f"{name} (test)")

    results[name] = {
        "val_pred": pred_val, "test_pred": pred_test,
        "val_rmse": val_rmse, "test_rmse": test_rmse,
    }
    trained_models[name] = model

    # 선형 모델은 선택된 alpha 출력
    if hasattr(model, 'alpha_'):
        info = f"  → best alpha={model.alpha_:.6g}"
        if hasattr(model, 'l1_ratio_'):
            info += f", l1_ratio={model.l1_ratio_:.2f}"
        print(info)

print(f"\n{'='*50}")
print(f"{len(results)}개 모델 학습 완료")

In [ ]:
# ─── RMSE 비교표 (Val / Test) ───
comparison = pd.DataFrame([
    {
        "Model": name,
        "Val RMSE": r["val_rmse"],
        "Test RMSE": r["test_rmse"],
    }
    for name, r in results.items()
]).sort_values("Val RMSE").reset_index(drop=True)
comparison.index += 1

comparison

## 5. Feature Importance (트리 모델 4종 비교)

LightGBM, XGBoost, CatBoost, RandomForest의 feature importance를 비교하여 공통 핵심 피처를 파악

In [ ]:
# ─── 트리 모델 4종 Feature Importance 추출 ───
tree_models = ["LightGBM", "XGBoost", "CatBoost", "RandomForest"]
importance_dict = {}

for name in tree_models:
    model = trained_models[name]
    imp = model.feature_importances_
    df = pd.DataFrame({"feature": X_train_s.columns, "importance": imp})
    # 정규화 (모델 간 스케일 통일)
    df["importance_norm"] = df["importance"] / df["importance"].sum()
    df = df.sort_values("importance", ascending=False).reset_index(drop=True)
    df.index += 1
    # 원본 feature명 추출 (X123_mean → X123)
    df["original_feature"] = df["feature"].str.replace(r"_(mean|std|min|max|range|median|skew)$", "", regex=True)
    df["agg_type"] = df["feature"].str.extract(r"_(mean|std|min|max|range|median|skew)$")[0]
    importance_dict[name] = df

# 모델별 Top 10 출력
for name in tree_models:
    df = importance_dict[name]
    print(f"\n{'='*50}")
    print(f"[{name}] Feature Importance Top 10")
    print(df[["feature", "importance", "importance_norm"]].head(10).to_string())
    zero_imp = (df["importance"] == 0).sum()
    print(f"중요도 = 0: {zero_imp}개 / {len(df)}개 ({zero_imp/len(df)*100:.1f}%)")

In [ ]:
# ─── 트리 모델 공통 중요 feature 분석 (원본 X번호 기준) ───
# 각 모델에서 원본 feature별 정규화 중요도 합산 → 4종 트리 모델 평균
orig_imp_per_model = {}
for name in tree_models:
    df = importance_dict[name]
    orig = df.groupby("original_feature")["importance_norm"].sum()
    orig_imp_per_model[name] = orig

orig_imp_all = pd.DataFrame(orig_imp_per_model)
orig_imp_all["mean_importance"] = orig_imp_all.mean(axis=1)
orig_imp_all = orig_imp_all.sort_values("mean_importance", ascending=False)

print("=" * 70)
print("원본 Feature 기준 중요도 Top 20 (4종 트리 모델 평균)")
print("=" * 70)
print(orig_imp_all.head(20).round(6).to_string())

# 시각화: 모델별 Top 20 비교
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 좌: 4종 트리 모델 평균 Top 20
top20 = orig_imp_all.head(20)
axes[0].barh(range(len(top20)), top20["mean_importance"], color="steelblue")
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20.index, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_title("원본 Feature 중요도 Top 20 (4종 평균)")
axes[0].set_xlabel("Normalized Importance (mean)")

# 우: 모델별 Top 20 히트맵
top20_models = orig_imp_all.head(20)[tree_models]
import seaborn as sns
sns.heatmap(top20_models, annot=True, fmt=".4f", cmap="YlOrRd",
            ax=axes[1], xticklabels=True, yticklabels=True)
axes[1].set_title("모델별 Feature 중요도 비교 (Top 20)")

plt.tight_layout()
plt.show()

## 6. 예측 분포 확인

In [ ]:
# ─── 최고 모델의 예측 분포 확인 ───
best_model_name = comparison.iloc[0]["Model"]
best_pred_val = results[best_model_name]["val_pred"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 실제 vs 예측 히스토그램
axes[0].hist(y_val, bins=50, alpha=0.7, label="actual", color="steelblue")
axes[0].hist(best_pred_val, bins=50, alpha=0.7, label="predicted", color="coral")
axes[0].set_title(f"Actual vs Predicted ({best_model_name})")
axes[0].legend()
axes[0].set_xlabel(TARGET_COL)

# 예측값 분포 확대
axes[1].hist(best_pred_val, bins=50, color="coral", edgecolor="black")
axes[1].set_title(f"Predicted Distribution ({best_model_name})")
axes[1].set_xlabel(TARGET_COL)

# Scatter
axes[2].scatter(y_val, best_pred_val, alpha=0.3, s=5, color="steelblue")
axes[2].plot([0, y_val.max()], [0, y_val.max()], "r--", linewidth=1)
axes[2].set_xlabel("Actual")
axes[2].set_ylabel("Predicted")
axes[2].set_title(f"Actual vs Predicted ({best_model_name})")

plt.tight_layout()
plt.show()

print(f"[{best_model_name}] 예측값 통계:")
print(f"  min={best_pred_val.min():.6f}, max={best_pred_val.max():.6f}, mean={best_pred_val.mean():.6f}")
print(f"  예측 0 비율: {(best_pred_val == 0).mean()*100:.1f}%")